# Slocum data exploration

Load a processed mission's **L0 / L1 / L2** NetCDF and poke at it — track,
sections, profiles, T–S, gridded fields, raw engineering channels.

Read-only. If the products don't exist yet, run
`mission_processing.ipynb` for that mission first.

| | dimension | contents |
|---|---|---|
| **L0** | `time` | every decoded sample, raw Slocum sensor names (`m_*`, `c_*`, `sci_*`), no QC |
| **L1** | `time` | CF names, TEOS-10 salinity & density, `profile_index`, deployment window |
| **L2** | `time × depth` | gridded |

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import gsw

try:
    import cmocean.cm as cmo
    CMAP = {"temperature": cmo.thermal, "potential_temperature": cmo.thermal,
            "salinity": cmo.haline, "conductivity": cmo.haline,
            "density": cmo.dense, "potential_density": cmo.dense,
            "chlorophyll": cmo.algae, "turbidity": cmo.turbid,
            "oxygen_concentration": cmo.oxy, "depth": cmo.deep}
except ImportError:  # cmocean is optional
    CMAP = {"temperature": "viridis", "potential_temperature": "viridis",
            "salinity": "plasma", "conductivity": "plasma",
            "density": "cividis", "potential_density": "cividis",
            "chlorophyll": "YlGn", "turbidity": "YlOrBr",
            "oxygen_concentration": "magma", "depth": "Blues"}

cmap = lambda var: CMAP.get(var, "viridis")
plt.rcParams["figure.figsize"] = (14, 5)

pd.set_option("display.max_rows", None)      # never truncate long variable tables
pd.set_option("display.max_colwidth", None)

## 1. Select the mission & load the datasets

Set `MISSION` to a mission number (or a `python/missions/` directory
prefix). The L0/L1/L2 files are expected under the mission's
`.../<NNN-name>/pyglider/L{0,1,2}/` — where `mission_processing.ipynb`
writes them.

In [ ]:
MISSION = 2

DATA_ROOT = Path("/Data/gfi/projects/slocum/data/delayed")

_token = f"{int(MISSION):03d}" if str(MISSION).isdigit() else str(MISSION)
_matches = [d for d in sorted(DATA_ROOT.glob(f"{_token}*")) if d.is_dir()]
assert len(_matches) == 1, f"expected one folder for {_token!r}, found {[d.name for d in _matches]}"
deployment_dir = _matches[0]
name = deployment_dir.name
pyglider_dir = deployment_dir / "pyglider"


def _load(level: str) -> xr.Dataset | None:
    f = pyglider_dir / level / f"{name}_{level}.nc"
    if not f.is_file():
        print(f"  {level}: not found ({f})")
        return None
    ds = xr.open_dataset(f)
    print(f"  {level}: {dict(ds.sizes)}")
    return ds


print(name)
l0 = _load("L0")
l1 = _load("L1")
l2 = _load("L2")

## 2. Overview

In [ ]:
def overview(ds: xr.Dataset, label: str) -> None:
    if ds is None:
        return
    t = pd.to_datetime(ds["time"].values)
    print(f"=== {label} ===")
    print(f"  time      : {t.min()}  →  {t.max()}   ({t.max() - t.min()})")
    print(f"  samples   : {ds.sizes.get('time', '—')}")
    for c in ("latitude", "longitude"):
        if c in ds:
            v = ds[c].values
            print(f"  {c:<10}: {np.nanmin(v):.3f} .. {np.nanmax(v):.3f}")
    if "depth" in ds:
        print(f"  depth     : {np.nanmin(ds['depth'].values):.1f} .. {np.nanmax(ds['depth'].values):.1f} m")
    if "profile_index" in ds:
        print(f"  profiles  : {int(np.nanmax(ds['profile_index'].values))}")
    print(f"  processing_level = {ds.attrs.get('processing_level')}   "
          f"featureType = {ds.attrs.get('featureType')}")
    print()


for ds, lbl in [(l0, "L0"), (l1, "L1"), (l2, "L2")]:
    overview(ds, lbl)

In [ ]:
def list_vars(ds: xr.Dataset, contains: str = "") -> pd.DataFrame:
    'Data variables in ds, optionally filtered by substring, with units + range.'
    rows = []
    for v in ds.data_vars:
        if contains and contains.lower() not in v.lower():
            continue
        a = ds[v].values
        num = np.issubdtype(a.dtype, np.number)
        rows.append({
            "variable": v,
            "dims": ",".join(ds[v].dims),
            "units": ds[v].attrs.get("units", ""),
            "long_name": ds[v].attrs.get("long_name", ""),
            "min": f"{np.nanmin(a):.4g}" if num and a.size else "",
            "max": f"{np.nanmax(a):.4g}" if num and a.size else "",
        })
    return pd.DataFrame(rows).set_index("variable")


list_vars(l1)

## 3. Track

Longitude / latitude coloured by time. (No coastline — add `cartopy` if you
want one.)

In [ ]:
ds = l1  # or l0
lon, lat = ds["longitude"].values, ds["latitude"].values
t = pd.to_datetime(ds["time"].values)
ok = np.isfinite(lon) & np.isfinite(lat)

fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(lon[ok], lat[ok], c=t[ok].astype("int64"), s=6, cmap="viridis")
ax.plot(lon[ok], lat[ok], "-", lw=0.4, alpha=0.4, color="grey")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_aspect("equal", "datalim")
ax.set_title(f"{name} — track")
cb = fig.colorbar(sc, ax=ax, shrink=0.7)
cb.ax.set_yticklabels([pd.Timestamp(v).strftime("%m-%d") for v in cb.get_ticks()])

## 4. Sections (depth–time)

`section(l1, "temperature")` — change the variable and re-run. Any numeric
`(time,)` variable in L1 (or L0) works.

In [ ]:
def section(ds: xr.Dataset, var: str, ax=None, *, clim=None, s=4):
    'Scatter section: time on x, depth on y, colour = var.'
    d = ds[var].values
    z = ds["depth"].values
    ok = np.isfinite(d) & np.isfinite(z)
    lo, hi = clim if clim else np.nanpercentile(d[ok], [1, 99])
    ax = ax or plt.subplots(figsize=(14, 5))[1]
    sc = ax.scatter(pd.to_datetime(ds["time"].values)[ok], z[ok], c=d[ok],
                    s=s, cmap=cmap(var), vmin=lo, vmax=hi)
    ax.invert_yaxis()
    ax.set_ylabel("depth [m]")
    ax.set_title(f"{var}   [{ds[var].attrs.get('units', '')}]")
    ax.figure.colorbar(sc, ax=ax, pad=0.01)
    return ax


fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
for ax, v in zip(axes, ["temperature", "salinity", "potential_density"]):
    section(l1, v, ax)
fig.tight_layout()

## 5. Profiles

`profile_index` in L1 counts up/down casts (`.5` = mid-turn). Plot a
selection of them, or all overlaid.

In [ ]:
def profiles(ds: xr.Dataset, var: str, which=None, ax=None, max_lines=40):
    'var vs depth for chosen profile numbers (default: an even sample).'
    pidx = ds["profile_index"].values
    whole = np.unique(pidx[np.isfinite(pidx) & (pidx % 1 == 0)]).astype(int)
    if which is None:
        step = max(1, len(whole) // max_lines)
        which = whole[::step]
    ax = ax or plt.subplots(figsize=(6, 9))[1]
    norm = plt.Normalize(min(which), max(which))
    for p in which:
        m = pidx == p
        ax.plot(ds[var].values[m], ds["depth"].values[m], lw=0.8,
                color=plt.cm.viridis(norm(p)))
    ax.invert_yaxis()
    ax.set_xlabel(f"{var} [{ds[var].attrs.get('units', '')}]")
    ax.set_ylabel("depth [m]")
    ax.set_title(f"{var} — profiles {which.min()}–{which.max()}")
    return ax


fig, axes = plt.subplots(1, 3, figsize=(15, 9))
for ax, v in zip(axes, ["temperature", "salinity", "potential_density"]):
    profiles(l1, v, ax=ax)
fig.tight_layout()

## 6. T–S diagram

Conservative Temperature vs Absolute Salinity (TEOS-10, via `gsw`), coloured
by depth, with σ₀ contours.

In [ ]:
ds = l1
p = ds["pressure"].values
SP = ds["salinity"].values          # practical salinity (pyglider output)
T = ds["temperature"].values        # in-situ temperature
lon = ds["longitude"].values
lat = ds["latitude"].values
z = ds["depth"].values

SA = gsw.SA_from_SP(SP, p, lon, lat)
CT = gsw.CT_from_t(SA, T, p)
ok = np.isfinite(SA) & np.isfinite(CT)

sa_g = np.linspace(np.nanpercentile(SA[ok], 0.5), np.nanpercentile(SA[ok], 99.5), 100)
ct_g = np.linspace(np.nanpercentile(CT[ok], 0.5), np.nanpercentile(CT[ok], 99.5), 100)
SIG = gsw.sigma0(*np.meshgrid(sa_g, ct_g))

fig, ax = plt.subplots(figsize=(8, 8))
cs = ax.contour(sa_g, ct_g, SIG, colors="grey", linewidths=0.7)
ax.clabel(cs, fmt="%.1f", fontsize=8)
sc = ax.scatter(SA[ok], CT[ok], c=z[ok], s=4, cmap=cmap("depth"))
ax.set_xlabel("Absolute Salinity  [g kg⁻¹]")
ax.set_ylabel("Conservative Temperature  [°C]")
ax.set_title(f"{name} — T–S")
fig.colorbar(sc, ax=ax, label="depth [m]")

## 7. Gridded (L2)

`(time, depth)` — pcolormesh / Hovmöller.

In [ ]:
if l2 is not None:
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    for ax, v in zip(axes, ["temperature", "salinity", "potential_density"]):
        l2[v].plot(ax=ax, x="time", y="depth", yincrease=False, robust=True, cmap=cmap(v))
        ax.set_title(v)
    fig.tight_layout()
else:
    print("no L2")

## 8. Raw engineering channels (L0)

Any raw sensor vs time — battery, pitch, ballast, vacuum, comms, ... Use
`list_vars(l0, "batt")` to search.

In [ ]:
list_vars(l0, "batt")

In [ ]:
def raw(sensor: str, ax=None, *, color_by_depth=True):
    'L0 raw sensor vs time; optionally colour by depth.'
    s = l0[sensor].values
    t = pd.to_datetime(l0["time"].values)
    ok = np.isfinite(s)
    ax = ax or plt.subplots(figsize=(14, 4))[1]
    if color_by_depth and "m_depth" in l0:
        z = l0["m_depth"].values
        sc = ax.scatter(t[ok], s[ok], c=z[ok], s=3, cmap=cmap("depth"))
        ax.figure.colorbar(sc, ax=ax, label="m_depth [m]")
    else:
        ax.plot(t[ok], s[ok], ".", ms=2)
    ax.set_title(f"{sensor}   [{l0[sensor].attrs.get('units', '')}]")
    ax.grid(alpha=0.3)
    return ax


for s in ["m_battery", "m_vacuum", "m_pitch", "m_ballast_pumped"]:
    if s in l0:
        raw(s, color_by_depth=False)
        plt.show()

## 9. Interactive (Plotly)

Zoom / pan / hover. Uses WebGL for the dense scatter and an OpenStreetMap
basemap (no token needed).

In [ ]:
import plotly.graph_objects as go

# --- L0 m_depth vs time — every sample, low opacity to show the spread ---
d = l0["m_depth"].values
t = pd.to_datetime(l0["time"].values)
ok = np.isfinite(d)

fig = go.Figure(go.Scattergl(
    x=t[ok], y=d[ok], mode="markers",
    marker=dict(size=2, color=d[ok], colorscale="Viridis", opacity=0.35,
                colorbar=dict(title="m_depth [m]")),
    hovertemplate="%{x|%Y-%m-%d %H:%M:%S}<br>%{y:.1f} m<extra></extra>",
))
fig.update_yaxes(autorange="reversed", title="m_depth [m]")
fig.update_xaxes(title="time")
fig.update_layout(title=f"{name} — L0 m_depth vs time",
                  height=550, template="plotly_white")
fig

In [ ]:
# --- GPS fixes on a map, shallower than 30 m ---
# m_gps_lat/lon are only set at surface fixes (~NaN underwater); dbdreader has
# already converted them to decimal degrees. Swap in m_lat/m_lon for the
# denser dead-reckoned track.
d = l0["m_depth"].values
lat = l0["m_gps_lat"].values
lon = l0["m_gps_lon"].values
tt = pd.to_datetime(l0["time"].values)
m = np.isfinite(lat) & np.isfinite(lon) & np.isfinite(d) & (d < 30)

fig = go.Figure(go.Scattermap(
    lat=lat[m], lon=lon[m], mode="markers+lines",
    marker=dict(size=8, color=tt[m].astype("int64"), colorscale="Viridis"),
    line=dict(width=1, color="rgba(80,80,80,0.4)"),
    text=[x.strftime("%Y-%m-%d %H:%M") for x in tt[m]],
    hovertemplate="%{lat:.4f}, %{lon:.4f}<br>%{text}<extra></extra>",
))
fig.update_layout(
    map=dict(style="open-street-map",
             center=dict(lat=float(np.nanmean(lat[m])), lon=float(np.nanmean(lon[m]))),
             zoom=8),
    height=650, margin=dict(l=0, r=0, t=30, b=0),
    title=f"{name} — GPS fixes (m_depth < 30 m), n={int(m.sum())}",
)
fig

## 10. Scratch

Your own cells below.